In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score
import json
import os

In [41]:
# ── CHANGE THESE TWO LINES per run ──────────────────────────────
MODEL_FOLDER = "xlm-roberta-base" # "bert-base-multilingual-cased" or "muril-base-cased" or "xlm-roberta-base"
STATE        =  "pretrained"  # "pretrained" or "finetuned"
# ────────────────────────────────────────────────────────────────

BASE_DIR   = "/Users/harshaggarwal/Projects_4/hinemo_project/models/hidden_states"
MODEL_DIR  = f"{BASE_DIR}/{MODEL_FOLDER}"
OUTPUT_DIR = f"/Users/harshaggarwal/Projects_4/hinemo_project/models/probing_results/{MODEL_FOLDER}"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Model:  {MODEL_FOLDER}")
print(f"State:  {STATE}")
print(f"Output: {OUTPUT_DIR}")

Model:  xlm-roberta-base
State:  pretrained
Output: /Users/harshaggarwal/Projects_4/hinemo_project/models/probing_results/xlm-roberta-base


In [42]:
hidden = np.load(f"{MODEL_DIR}/hidden_{STATE}.npy")
meta   = pd.read_csv(f"{MODEL_DIR}/metadata.csv")

lambda_vals = meta["lambda"].values

n_samples, n_layers, hidden_dim = hidden.shape

print(f"Hidden states shape: {hidden.shape}")
print(f"Lambda values shape: {lambda_vals.shape}")
print(f"Lambda range: {lambda_vals.min():.3f} → {lambda_vals.max():.3f}")
print(f"Lambda mean:  {lambda_vals.mean():.3f}")

Hidden states shape: (3000, 12, 768)
Lambda values shape: (3000,)
Lambda range: 0.000 → 1.000
Lambda mean:  0.425


In [43]:
r2_per_layer = []

for layer in range(n_layers):
    X = hidden[:, layer, :]   # shape: (3000, 768) — representations at this layer
    y = lambda_vals            # shape: (3000,)     — λ values to predict

    probe  = Ridge(alpha=1.0)
    scores = cross_val_score(probe, X, y, cv=5, scoring="r2")
    mean_r2 = scores.mean()

    r2_per_layer.append(mean_r2)
    print(f"Layer {layer+1:2d}: R² = {mean_r2:.4f}")

Layer  1: R² = 0.6561
Layer  2: R² = 0.5904
Layer  3: R² = 0.5515
Layer  4: R² = 0.4107
Layer  5: R² = 0.5516
Layer  6: R² = 0.1859
Layer  7: R² = -0.1249
Layer  8: R² = 0.0833
Layer  9: R² = 0.1097
Layer 10: R² = 0.3580
Layer 11: R² = 0.5921
Layer 12: R² = 0.7005


In [44]:
LSL = int(np.argmax(r2_per_layer)) + 1   # +1 because layers are 1-indexed in the paper

print(f"\nLSL ({MODEL_FOLDER}, {STATE}) = Layer {LSL}")
print(f"Peak R²                       = {r2_per_layer[LSL-1]:.4f}")
print(f"\nFull R² curve:")
for i, r2 in enumerate(r2_per_layer):
    marker = " ← LSL" if i+1 == LSL else ""
    print(f"  Layer {i+1:2d}: {r2:.4f}{marker}")


LSL (xlm-roberta-base, pretrained) = Layer 12
Peak R²                       = 0.7005

Full R² curve:
  Layer  1: 0.6561
  Layer  2: 0.5904
  Layer  3: 0.5515
  Layer  4: 0.4107
  Layer  5: 0.5516
  Layer  6: 0.1859
  Layer  7: -0.1249
  Layer  8: 0.0833
  Layer  9: 0.1097
  Layer 10: 0.3580
  Layer 11: 0.5921
  Layer 12: 0.7005 ← LSL


In [45]:
results = {
    "model"       : MODEL_FOLDER,
    "state"       : STATE,
    "r2_per_layer": r2_per_layer,
    "LSL"         : LSL,
    "peak_r2"     : r2_per_layer[LSL-1],
}

save_path = f"{OUTPUT_DIR}/probing_{STATE}.json"
with open(save_path, "w") as f:
    json.dump(results, f, indent=2)

print(f"Results saved to {save_path}")

Results saved to /Users/harshaggarwal/Projects_4/hinemo_project/models/probing_results/xlm-roberta-base/probing_pretrained.json


In [48]:
import json
import os

models = [
    "bert-base-multilingual-cased",
    "muril-base-cased", 
    "xlm-roberta-base"
]
states = ["pretrained", "finetuned"]

print(f"{'Model':<35} {'State':<12} {'LSL':<6} {'Peak R²'}")
print("-" * 65)

for model in models:
    for state in states:
        path = f"/Users/harshaggarwal/Projects_4/hinemo_project/models/probing_results/{model}/probing_{state}.json"
        if os.path.exists(path):
            with open(path) as f:
                r = json.load(f)
            print(f"{model:<35} {state:<12} {r['LSL']:<6} {r['peak_r2']:.4f}")
        else:
            print(f"{model:<35} {state:<12} {'--':<6} --  (not yet run)")

Model                               State        LSL    Peak R²
-----------------------------------------------------------------
bert-base-multilingual-cased        pretrained   5      0.7139
bert-base-multilingual-cased        finetuned    4      0.6970
muril-base-cased                    pretrained   4      0.7615
muril-base-cased                    finetuned    3      0.7120
xlm-roberta-base                    pretrained   12     0.7005
xlm-roberta-base                    finetuned    1      0.6525


In [51]:
import json
with open("/Users/harshaggarwal/Projects_4/hinemo_project/models/checkpoints/xlm-roberta-base/best_model/config.json") as f:
    config = json.load(f)
print(config["model_type"])
print(config["architectures"])

xlm-roberta
['XLMRobertaForSequenceClassification']


In [52]:
for cv_folds in [3, 5, 10]:
    scores = cross_val_score(Ridge(alpha=1.0), hidden[:, 0, :], lambda_vals, 
                             cv=cv_folds, scoring="r2")
    print(f"CV={cv_folds}: R² = {scores.mean():.4f} ± {scores.std():.4f}")

CV=3: R² = 0.6468 ± 0.0524
CV=5: R² = 0.6561 ± 0.0471
CV=10: R² = 0.6503 ± 0.0558


In [53]:
print(meta.columns.tolist())

['id', 'gpt_emotion', 'lambda']


In [55]:
val_df   = pd.read_csv("/Users/harshaggarwal/Projects_4/hinemo_project/dataset/data/final_splits(524randomseed)/hinemo_val.csv")

In [56]:
# Check 2 — is lambda correlated with text length?
# merge metadata with val_df on id to get text column
merged = meta.merge(val_df[["id", "text"]], on="id")
merged["text_length"] = merged["text"].str.split().str.len()
print(merged[["lambda", "text_length"]].corr())

               lambda  text_length
lambda       1.000000    -0.029442
text_length -0.029442     1.000000
